In [11]:
# Libaries
import matplotlib.pyplot as plt
import scienceplots
from matplotlib.animation import FuncAnimation
from matplotlib.gridspec import GridSpec
plt.style.use(["science", "no-latex"])
import torch
import gpytorch
torch.set_default_dtype(torch.float64)
import numpy as np
import warnings
import os
import cv2
import numpy as np
import shutil



# SmallsatSim stuff
from smallsat_sim.utils.logger import Logger
import matplotlib.pyplot as plt

# Zero Order GPMPC package stuff
from zero_order_gpmpc.models.gpytorch_models.gpytorch_gp import (
    BatchIndependentMultitaskGPModel,
)

In [12]:
# Create Logger object (needed to load data)
logger = Logger()

# Load the pandas dataframe
df = logger.load_log()

# Print some information about the df
df.shape
df.info()
df.describe()

Loading log files from directory: /media/ahansson/2884-8728/logs/2024-10-17_18-53-26
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102 entries, 0 to 101
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   RunID           102 non-null    int64  
 1   Timestamp       102 non-null    float64
 2   gp_GT           102 non-null    object 
 3   gp_pred         102 non-null    object 
 4   gp_lower        102 non-null    object 
 5   gp_upper        102 non-null    object 
 6   n_points_GP     102 non-null    int64  
 7   tracking_error  102 non-null    float64
 8   attitude_error  102 non-null    float64
 9   solve_time      102 non-null    int64  
 10  mpc_cost        102 non-null    object 
 11  u_demanded      102 non-null    object 
 12  velocity        102 non-null    float64
 13  u_actual        102 non-null    object 
 14  e_gp            100 non-null    float64
 15  e_nom           100 non-null    float64


,RunID,Timestamp,n_points_GP,tracking_error,attitude_error,solve_time,velocity,e_gp,e_nom
count,102.000000,102.000000,102.000000,102.000000,102.000000,102.0,102.000000,100.000000,100.000000
mean,0.500000,2.500000,21.941176,0.191839,39.017879,0.0,0.127911,0.000221,0.000223
std,0.502469,1.479229,11.815264,0.158944,9.400990,0.0,0.058497,0.000137,0.000137
min,0.000000,0.000000,0.000000,0.016855,20.965414,0.0,0.000000,0.000037,0.000040
25%,0.000000,1.225000,12.250000,0.070407,30.998712,0.0,0.084445,0.000112,0.000114
50%,0.500000,2.500000,22.500000,0.140236,39.552197,0.0,0.144041,0.000196,0.000198
75%,1.000000,3.775000,32.000000,0.288151,47.343627,0.0,0.176675,0.000303,0.000306
max,1.000000,5.000000,40.000000,0.522211,54.140132,0.0,0.197749,0.000564,0.000564


In [13]:
# Calculate and print the mean tracking error of all run ids
mean_tracking_error = df.groupby("run_id")["tracking_error"].mean()




Mean Tracking Error for RunID 0: 0.2732836502237615
Mean Tracking Error for RunID 1: 0.11039479645736908


In [14]:
mean_e_gp_runID_0 = df[df['RunID'] == 0]['e_gp'].mean()
mean_e_gp_runID_1 = df[df['RunID'] == 1]['e_gp'].mean()

mean_e_nom_runID_0 = df[df['RunID'] == 0]['e_nom'].mean()
mean_e_nom_runID_1 = df[df['RunID'] == 1]['e_nom'].mean()

print(f"Mean e_gp for RunID 0: {mean_e_gp_runID_0}")
print(f"Mean e_gp for RunID 1: {mean_e_gp_runID_1}")
print(f"Mean e_nom for RunID 0: {mean_e_nom_runID_0}")
print(f"Mean e_nom for RunID 1: {mean_e_nom_runID_1}")

Mean e_gp for RunID 0: 0.0002236001854549304
Mean e_gp for RunID 1: 0.00021938875343816764
Mean e_nom for RunID 0: 0.0002258139891225751
Mean e_nom for RunID 1: 0.00022012913536387604
